# Cattle Re-ID — Runner de Colab (replicación del paper, Fases 0→3)

Orquestador para correr la **replicación del paper (VGG16_BN)** en **Google Colab (Pro) con GPU**. La lógica vive en `src/` y `scripts/`; este notebook solo orquesta (ver `CLAUDE.md`), igual que `kaggle_runner.ipynb`. El backbone ResNet-50 (Fase 4) es aparte y no se corre acá.

**Antes de correr:**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU* (con Colab Pro: preferir A100/V100 si están disponibles; T4 si no).
2. Subí el dataset **una sola vez** a Google Drive como un `.zip` (ver sección 3). No hace falta resubirlo en cada sesión.
3. Los pesos ImageNet **no** hace falta subirlos: Colab tiene internet, así que `models.py` los baja automáticamente de torchvision si no los encuentra localmente.

**Resiliencia a desconexiones (Colab corta sesiones idle/largas):**
- `outputs/checkpoints/`, `outputs/logs/` y `outputs/results/` quedan **symlinkeados a Drive** (sección 5), así que sobreviven a un reinicio del entorno de ejecución.
- `scripts/02_train_vgg.py` ahora guarda **progreso incremental** (`02_vgg_summary.progress.json`) después de cada corrida (variante × semilla) y, si lo volvés a correr, **saltea las corridas ya completadas**. Si se corta a mitad de una corrida puntual, esa corrida (no el sweep entero) se reinicia desde cero — pero como la seed queda fija, da exactamente el mismo resultado que si no se hubiera cortado.
- Para ignorar el progreso guardado y arrancar de cero: `python scripts/02_train_vgg.py --fresh`.


## 0. Verificar GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No hay GPU: Entorno de ejecución -> Cambiar tipo de entorno -> GPU.'
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print(f'GPU: {name} | capability sm_{cap[0]}{cap[1]}')


## 1. Montar Google Drive

Acá vamos a persistir el dataset (zip subido una vez) y los outputs (checkpoints/logs/results) para que sobrevivan a una desconexión. Ajustá `DRIVE_ROOT` si preferís otra carpeta dentro de tu Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/cattle_reid')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('DRIVE_ROOT:', DRIVE_ROOT)


## 2. Traer el código (git clone / pull)

In [ ]:
import os

REPO_URL = 'https://github.com/benjaminvitale/tp-final-vision2-Pujol-Vitale.git'
REPO_DIR = '/content/tp-final-vision2-Pujol-Vitale'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only

%cd {REPO_DIR}
!git log --oneline -1


## 3. Dataset — subir una vez a Drive, extraer localmente cada sesión

`/content` es disco local del entorno de ejecución (rápido pero efímero); Drive es persistente pero lento para miles de archivos chicos vía el mount FUSE. Por eso: el dataset se sube **una vez** como `.zip` a Drive, y en cada sesión se copia y descomprime al disco local (`data_local/`, que es justo donde `config.py` ya lo busca — no hace falta tocar código ni setear variables de entorno).

**Paso único (fuera de Colab):** comprimí tu carpeta local `datasets/BeefCattle_Muzzle_Individualized/` en un `.zip` (en Windows: clic derecho → Enviar a → Carpeta comprimida, o `Compress-Archive` en PowerShell) y subilo a tu Drive en:

```
MyDrive/cattle_reid/BeefCattle_Muzzle_Individualized.zip
```

Si lo subiste con otro nombre/ruta, ajustá `DRIVE_DATASET_ZIP` abajo.


In [ ]:
DRIVE_DATASET_ZIP = DRIVE_ROOT / 'BeefCattle_Muzzle_Individualized.zip'
LOCAL_DATA_DIR = Path(REPO_DIR) / 'data_local'

assert DRIVE_DATASET_ZIP.is_file(), (
    f'No encuentro {DRIVE_DATASET_ZIP}. Subi el zip del dataset a tu Drive primero '
    '(ver celda de arriba) o ajusta DRIVE_DATASET_ZIP.'
)

extracted = LOCAL_DATA_DIR / 'BeefCattle_Muzzle_Individualized'
if not extracted.is_dir():
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    print('Extrayendo dataset a disco local (una vez por sesión)...')
    !unzip -q "{DRIVE_DATASET_ZIP}" -d "{LOCAL_DATA_DIR}"
else:
    print('Dataset ya extraido en esta sesion:', extracted)

n_classes = len(list(extracted.iterdir())) if extracted.is_dir() else 0
print('Carpeta dataset:', extracted, '| subcarpetas (clases):', n_classes)


## 4. Persistir outputs en Drive (checkpoints / logs / results)

Symlinkeamos `outputs/checkpoints/`, `outputs/logs/` y `outputs/results/` a carpetas en Drive. Así, si el entorno de Colab se desconecta y hay que reabrir el notebook, los checkpoints y el progreso del sweep **ya están ahí** (no se pierde nada salvo la corrida puntual que estaba a mitad de camino).

La primera vez copia a Drive lo que venga versionado en git (p.ej. `outputs/results/00_inspect_report.json`) sin pisar progreso previo; las veces siguientes no toca lo que ya esté en Drive.


In [ ]:
import shutil

DRIVE_OUTPUTS = DRIVE_ROOT / 'outputs'

for sub in ('checkpoints', 'logs', 'results'):
    drive_sub = DRIVE_OUTPUTS / sub
    drive_sub.mkdir(parents=True, exist_ok=True)
    local_sub = Path(REPO_DIR) / 'outputs' / sub

    if local_sub.exists() and not local_sub.is_symlink():
        for item in local_sub.iterdir():
            target = drive_sub / item.name
            if not target.exists():
                if item.is_dir():
                    shutil.copytree(item, target)
                else:
                    shutil.copy2(item, target)
        shutil.rmtree(local_sub)

    if not local_sub.exists():
        os.symlink(drive_sub, local_sub, target_is_directory=True)

    print(f'outputs/{sub} -> {drive_sub}')


## 5. Verificar rutas (DATA_DIR + PRETRAINED_DIR)

`config.py` resuelve el dataset desde `data_local/` (recién extraído) y los pesos ImageNet desde `pretrained_weights/` si existieran localmente; si no los encuentra, `models.py` los baja de torchvision (Colab tiene internet, a diferencia de Kaggle con Internet OFF).


In [ ]:
import config, importlib
importlib.reload(config)

print('DATA_DIR       :', config.DATA_DIR, '| existe:', config.DATA_DIR.is_dir())
print('PRETRAINED_DIR :', config.PRETRAINED_DIR, '(None = se baja de torchvision, OK en Colab)')

assert config.DATA_DIR.is_dir(), 'DATA_DIR no encontrado: revisa la seccion 3 (extraccion del dataset).'


## 6. Fase 0 — Inspección de datos (SIEMPRE primero)

Sanity check contra el paper: **268 clases, 4923 imágenes, 4–70 por clase**. No avanzar si no cuadra.


In [ ]:
!python scripts/00_inspect_data.py


## 7. Splits (ya commiteados — NO regenerar)

Los splits viven en `outputs/splits/` con rutas relativas a `DATA_DIR` (vienen del repo, no de Drive), así que el mismo split sirve en Kaggle, Colab y local. Esta celda solo los **verifica**.


In [ ]:
from src.utils import load_json
for name in ('train', 'val', 'test'):
    n = len(load_json(config.SPLITS_DIR / f'{name}.json'))
    print(f'{name:5}: {n} imagenes')
label_map = load_json(config.SPLITS_DIR / 'label_map.json')
print('clases en label_map:', len(label_map))


## 7b. Precomputar augmentation (imágenes sintéticas a disco)

Igual que en Kaggle: genera una vez el set sintético para la variante `ce_aug`/`wce_aug`, en disco local (`outputs/aug_cache/`, no symlinkeado a Drive — es regenerable y rápido, no hace falta persistirlo).


In [ ]:
!python scripts/04_precompute_aug.py


## 8. Smoke test en GPU (validar ANTES de quemar cuota)

Confirma que paths + GPU + pipeline resuelven end-to-end. Subset chico, 2 épocas, `pretrained=False`. La accuracy va a dar ~0: es esperado, valida el pipeline, no la ciencia.


In [ ]:
!python scripts/02_train_vgg.py --smoke


## 9. Fase 3 — Replicación VGG16_BN (el deliverable del paper)

3 variantes (`ce`, `ce_aug`, `wce`) × 3 semillas, 50 épocas — la misma receta que en `kaggle_runner.ipynb` (ver `config.py` / `DEVIATIONS.md` para el porqué de 3 y no 5 semillas).

**Si la sesión se corta:** simplemente volvé a correr esta misma celda. Gracias a la sección 4 (outputs symlinkeados a Drive) y al progreso incremental de `02_train_vgg.py`, las corridas ya completadas se saltean automáticamente y el sweep continúa donde quedó.


In [ ]:
# 3 semillas para entrar en el presupuesto de horas en GPU (ver DEVIATIONS.md D2).
# Para las 5 del plan: --seeds 0 1 2 3 4 (puede no entrar en una sola sesión, ver nota de arriba).
!python scripts/02_train_vgg.py --seeds 0 1 2


## 10. Resultados y checkpoints (ya persistidos en Drive vía symlink)

In [ ]:
import json

print('=== outputs/results/ ===')
!ls -lh outputs/results/
print('\n=== outputs/checkpoints/ ===')
!ls -lh outputs/checkpoints/

p = config.RESULTS_DIR / '02_vgg_summary.json'
if p.exists():
    print('\n=== 02_vgg_summary.json (resumen replicacion) ===')
    data = json.loads(p.read_text())
    print(json.dumps(data.get('summary', data), indent=2, ensure_ascii=False)[:2500])
